In [1]:
search_sample_size = 20_000
rf_n_iter = 8
models_to_run = ["logreg", "random_forest"]
run_temporal = True
save_to_mongo = True
reuse_results = False
cache_name = "results.joblib"
rf_trees_temporal = 100
run_leakage = True

In [2]:
import sys; sys.path.append("..")
import config
config.SEARCH_SAMPLE_SIZE = search_sample_size
config.RF_N_ITER = rf_n_iter

import joblib
import pandas as pd
from src.db_sql import load_clean_data
from src.features import get_xy, split_data
from src.models import tune_and_fit, fit_baseline
from src.evaluation import evaluate_model
from src.temporal import temporal_split, fit_temporal
from src.db_mongo import save_params, save_metrics, save_predictions, save_leakage
from src.logger_setup import flush_mongo_logs

from src.leakage_check import leakage_check

df = load_clean_data()
X, y = get_xy(df)
X_train, X_test, y_train, y_test = split_data(X, y)

if run_leakage:
    leak = leakage_check(df)
    print(leak[["variante", "accuracy", "f1_macro", "delta_vs_modelo_real"]].round(3))
    if save_to_mongo:
        save_leakage(leak)
flush_mongo_logs()


2026-11-10 02:22:59,871 | INFO     | src.db_sql | Tabla 'datos_limpios' leída: 174116 filas, 20 columnas


2026-11-10 02:22:59,923 | INFO     | src.features | X: (174116, 12) | y: (174116,) | predictoras: ['M2', 'AMBIENTES', 'ANTIGUEDAD', 'BAÑOS', 'BAULERA', 'COCHERA', 'LAVADERO', 'TERRAZA', 'LAT', 'LON', 'BARRIO', 'ORIENT']


2026-11-10 02:23:00,587 | INFO     | src.features | Train: 139292 filas | Test: 34824 filas


2026-11-10 02:23:00,596 | INFO     | src.features | Target en train:
SEGMENTO
alto     0.333
medio    0.333
bajo     0.333


2026-11-10 02:23:00,601 | INFO     | src.features | Target en test:
SEGMENTO
alto     0.333
medio    0.333
bajo     0.333


2026-11-10 02:23:00,745 | INFO     | src.leakage_check | Leakage check: muestra de 40000 filas (train 32000, test 8000)


... (3 outputs left out)

2026-11-10 02:23:37,671 | INFO     | src.leakage_check | Leakage check [con DOLARES + AÑO]: accuracy=0.709 f1_macro=0.705


2026-11-10 02:23:40,969 | INFO     | src.leakage_check | Leakage check [con PRECIO_M2 + AÑO]: accuracy=0.855 f1_macro=0.855


                    variante  accuracy  f1_macro  delta_vs_modelo_real
0  sin DOLARES (modelo real)     0.647     0.644                 0.000
1       sin DOLARES, con AÑO     0.648     0.644                 0.001
2                con DOLARES     0.689     0.685                 0.042
3          con DOLARES + AÑO     0.708     0.705                 0.061
4        con PRECIO_M2 + AÑO     0.855     0.855                 0.208


2026-11-10 02:23:42,012 | INFO     | src.db_mongo | Mongo 'leakage' [leakage]: 5 documentos insertados (run_id=20261110_022250)


In [3]:
# entrenamiento con caché, para no repetir los 20 minutos del Random Forest:
config.MODELS_DIR.mkdir(exist_ok=True)
RES_PATH = config.MODELS_DIR / cache_name
if reuse_results and RES_PATH.exists():
    results = joblib.load(RES_PATH)
    results = {k: v for k, v in results.items() if k in models_to_run + ["dummy"]}
else:
    results = {n: tune_and_fit(n, X_train, y_train) for n in models_to_run}
    results["dummy"] = fit_baseline(X_train, y_train)
    joblib.dump(results, RES_PATH)

2026-11-10 02:23:42,440 | INFO     | src.models | [logreg] búsqueda de hiperparámetros sobre 20000 filas (GridSearchCV, cv=3, scoring=f1_macro)


2026-11-10 02:23:59,765 | INFO     | src.models | [logreg] mejores parámetros: {'clf__C': 1} | f1_macro en CV: 0.6106 | 17 s


2026-11-10 02:24:16,303 | INFO     | src.models | [logreg] reentrenado con 139292 filas en 17 s


2026-11-10 02:24:16,656 | INFO     | src.models | [random_forest] búsqueda de hiperparámetros sobre 20000 filas (RandomizedSearchCV, cv=3, scoring=f1_macro)


2026-11-10 02:30:02,096 | INFO     | src.models | [random_forest] mejores parámetros: {'clf__n_estimators': 200, 'clf__min_samples_leaf': 5, 'clf__max_features': 0.5, 'clf__max_depth': 20} | f1_macro en CV: 0.6384 | 345 s


2026-11-10 02:42:14,894 | INFO     | src.models | [random_forest] reentrenado con 139292 filas en 733 s


2026-11-10 02:42:17,706 | INFO     | src.models | [dummy] estrategia=stratified | f1_macro en CV: 0.3328


In [4]:
# escenario aleatorio

all_metrics, all_preds = [], []
for name, r in results.items():
    m, p = evaluate_model(name, r["model"], X_train, y_train, X_test, y_test)
    all_metrics.append(m); all_preds.append(p)

resumen = pd.DataFrame(all_metrics)[["model", "accuracy", "precision_macro", "recall_macro",
                                     "f1_macro", "accuracy_train_sample"]].round(4)
print(resumen)

2026-11-10 02:42:29,450 | INFO     | src.evaluation | [logreg] TEST accuracy=0.6189 precision=0.6131 recall=0.6189 f1_macro=0.6151 (train acc 0.6222)


2026-11-10 02:42:29,452 | INFO     | src.evaluation | [logreg] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[8191 2560  856]
 [3316 5216 3075]
 [ 898 2565 8147]]


2026-11-10 02:42:34,524 | WARNING  | src.evaluation | [random_forest] posible sobreajuste: accuracy train 0.800 vs test 0.678


2026-11-10 02:42:34,878 | INFO     | src.evaluation | [random_forest] TEST accuracy=0.6784 precision=0.6777 recall=0.6784 f1_macro=0.6780 (train acc 0.8002)


2026-11-10 02:42:34,879 | INFO     | src.evaluation | [random_forest] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[8639 2452  516]
 [2668 6454 2485]
 [ 520 2559 8531]]


2026-11-10 02:42:38,165 | INFO     | src.evaluation | [dummy] TEST accuracy=0.3325 precision=0.3325 recall=0.3325 f1_macro=0.3325 (train acc 0.3324)


2026-11-10 02:42:38,166 | INFO     | src.evaluation | [dummy] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[3921 3872 3814]
 [3811 3851 3945]
 [3941 3862 3807]]


           model  accuracy  precision_macro  recall_macro  f1_macro  \
0         logreg    0.6189           0.6131        0.6189    0.6151   
1  random_forest    0.6784           0.6777        0.6784    0.6780   
2          dummy    0.3325           0.3325        0.3325    0.3325   

   accuracy_train_sample  
0                 0.6222  
1                 0.8002  
2                 0.3324  


In [5]:
# escenario temporal

t_metrics, t_preds = [], []
if run_temporal:
    Xtr_t, Xte_t, ytr_t, yte_t = temporal_split(df, X, y)
    for name, r in results.items():
        mdl = fit_temporal(name, r["model"], Xtr_t, ytr_t, rf_trees=rf_trees_temporal)
        m, p = evaluate_model(name, mdl, Xtr_t, ytr_t, Xte_t, yte_t)
        t_metrics.append(m); t_preds.append(p)

    print(pd.DataFrame(t_metrics)[["model", "accuracy", "precision_macro",
                                   "recall_macro", "f1_macro"]].round(4))
    pt = pd.concat(t_preds)
    pt["AÑO"] = df.loc[pt["id_fila"], "AÑO"].to_numpy()
    print(pt.groupby(["modelo", "AÑO"])["acierto"].mean().unstack().round(3))

2026-11-10 02:42:38,455 | INFO     | src.temporal | Split temporal: train 145850 filas (2001-2013) | test 28266 filas (2014-2016)


2026-11-10 02:42:55,290 | INFO     | src.temporal | [logreg] ajuste temporal con 145850 filas en 17 s


2026-11-10 02:42:57,188 | INFO     | src.evaluation | [logreg] TEST accuracy=0.6078 precision=0.6016 recall=0.6078 f1_macro=0.6037 (train acc 0.6190)


2026-11-10 02:42:57,190 | INFO     | src.evaluation | [logreg] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[6645 2036  740]
 [2667 4123 2631]
 [ 855 2156 6413]]


2026-11-10 02:49:46,903 | INFO     | src.temporal | [random_forest] ajuste temporal con 145850 filas en 410 s


2026-11-10 02:49:48,894 | WARNING  | src.evaluation | [random_forest] posible sobreajuste: accuracy train 0.796 vs test 0.641


... (3 outputs left out)

2026-11-10 02:49:51,477 | INFO     | src.evaluation | [dummy] TEST accuracy=0.3334 precision=0.3334 recall=0.3334 f1_macro=0.3334 (train acc 0.3333)


2026-11-10 02:49:51,478 | INFO     | src.evaluation | [dummy] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[3185 3175 3061]
 [3138 3103 3180]
 [3137 3150 3137]]


           model  accuracy  precision_macro  recall_macro  f1_macro
0         logreg    0.6078           0.6016        0.6078    0.6037
1  random_forest    0.6411           0.6414        0.6411    0.6413
2          dummy    0.3334           0.3334        0.3334    0.3334


AÑO             2014   2015   2016
modelo                            
dummy          0.332  0.336  0.330
logreg         0.616  0.615  0.563
random_forest  0.656  0.647  0.581


In [6]:
# persistencia

if save_to_mongo:
    save_params(results)
    save_metrics(all_metrics, "aleatorio")
    save_predictions(pd.concat(all_preds, ignore_index=True), "aleatorio")
    if run_temporal:
        save_metrics(t_metrics, "temporal")
        save_predictions(pd.concat(t_preds, ignore_index=True), "temporal")
flush_mongo_logs()

2026-11-10 02:49:58,025 | INFO     | src.db_mongo | Mongo 'parametros' [comun]: 3 documentos insertados (run_id=20261110_022250)


2026-11-10 02:49:58,840 | INFO     | src.db_mongo | Mongo 'metricas' [aleatorio]: 3 documentos insertados (run_id=20261110_022250)


2026-11-10 02:51:36,083 | INFO     | src.db_mongo | Mongo 'predicciones' [aleatorio]: 104472 documentos insertados (run_id=20261110_022250)


2026-11-10 02:51:37,383 | INFO     | src.db_mongo | Mongo 'metricas' [temporal]: 3 documentos insertados (run_id=20261110_022250)


2026-11-10 02:52:31,583 | INFO     | src.db_mongo | Mongo 'predicciones' [temporal]: 84798 documentos insertados (run_id=20261110_022250)


In [7]:
# prueba leakage

all_metrics, all_preds = [], []
for name, r in results.items():
    m, p = evaluate_model(name, r["model"], X_train, y_train, X_test, y_test)
    all_metrics.append(m); all_preds.append(p)

resumen = pd.DataFrame(all_metrics)[["model", "accuracy", "precision_macro", "recall_macro",
                                     "f1_macro", "accuracy_train_sample"]].round(4)
print(resumen)

print(leakage_check(df))
flush_mongo_logs()

2026-11-10 02:52:34,076 | INFO     | src.evaluation | [logreg] TEST accuracy=0.6189 precision=0.6131 recall=0.6189 f1_macro=0.6151 (train acc 0.6222)


2026-11-10 02:52:34,077 | INFO     | src.evaluation | [logreg] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[8191 2560  856]
 [3316 5216 3075]
 [ 898 2565 8147]]


2026-11-10 02:52:37,252 | WARNING  | src.evaluation | [random_forest] posible sobreajuste: accuracy train 0.800 vs test 0.678


2026-11-10 02:52:37,447 | INFO     | src.evaluation | [random_forest] TEST accuracy=0.6784 precision=0.6777 recall=0.6784 f1_macro=0.6780 (train acc 0.8002)


2026-11-10 02:52:37,448 | INFO     | src.evaluation | [random_forest] matriz de confusión (filas=real, columnas=predicho ['bajo', 'medio', 'alto']):
[[8639 2452  516]
 [2668 6454 2485]
 [ 520 2559 8531]]


2026-11-10 02:52:40,038 | INFO     | src.evaluation | [dummy] TEST accuracy=0.3325 precision=0.3325 recall=0.3325 f1_macro=0.3325 (train acc 0.3324)


... (5 outputs left out)

2026-11-10 02:53:19,281 | INFO     | src.leakage_check | Leakage check [con DOLARES]: accuracy=0.689 f1_macro=0.685


2026-11-10 02:53:22,948 | INFO     | src.leakage_check | Leakage check [con DOLARES + AÑO]: accuracy=0.709 f1_macro=0.705


2026-11-10 02:53:26,199 | INFO     | src.leakage_check | Leakage check [con PRECIO_M2 + AÑO]: accuracy=0.855 f1_macro=0.855


                    variante predictoras_extra  accuracy  f1_macro  \
0  sin DOLARES (modelo real)                []  0.647375  0.643659   
1       sin DOLARES, con AÑO             [AÑO]  0.648250  0.644273   
2                con DOLARES         [DOLARES]  0.689250  0.685265   
3          con DOLARES + AÑO    [DOLARES, AÑO]  0.708500  0.705051   
4        con PRECIO_M2 + AÑO  [PRECIO_M2, AÑO]  0.855125  0.854914   

   filas_train  filas_test  delta_vs_modelo_real  
0        32000        8000              0.000000  
1        32000        8000              0.000875  
2        32000        8000              0.041875  
3        32000        8000              0.061125  
4        32000        8000              0.207750  


In [8]:
# Borrado
# from src.db_mongo import get_db
# db = get_db()
# for c in ("predicciones", "metricas", "parametros"):
#     db[c].drop()